# EDA y Medidas de Calidad con DuckDB

**Objetivo:** Realizar un análisis exploratorio de datos (EDA) y calcular medidas de calidad sobre el dataset `sales_raw.csv`, identificando problemas de calidad y preparando las métricas necesarias.

**Dataset:** `data/sales_raw.csv` con errores intencionados:
- Canales inconsistentes
- Decimales con coma
- Valores nulos
- Unidades negativas
- Fechas inválidas (2099)
- Duplicados

**Entorno:** conda sdb-lab

## 1. Configuración e importación de librerías

In [1]:
# Importar librerías necesarias
import pandas as pd
import numpy as np
import json
import duckdb
from pathlib import Path
from datetime import datetime
from collections import OrderedDict

# Configurar pandas para mejor visualización
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

print(f"Pandas version: {pd.__version__}")
print(f"DuckDB version: {duckdb.__version__}")

Pandas version: 2.3.3
DuckDB version: 1.4.1


## 2. Cargar datos crudos

In [5]:
import os

path = os.getcwd()
print(f"Directorio de trabajo actual: {path}")
# Definir rutas
BASE = Path.cwd()
DATA = BASE / "data"
RAW = DATA / "sales_raw.csv"

print(f"Directorio base: {BASE}")
print(f"Archivo de datos: {RAW}")
print(f"Archivo existe: {RAW.exists()}")

Directorio de trabajo actual: /home/jmsa/IESRafaelAlberti/RafaelAlberti25_26/Modulos/SBD/UD1
Directorio base: /home/jmsa/IESRafaelAlberti/RafaelAlberti25_26/Modulos/SBD/UD1
Archivo de datos: /home/jmsa/IESRafaelAlberti/RafaelAlberti25_26/Modulos/SBD/UD1/data/sales_raw.csv
Archivo existe: False


In [3]:
# Leer el CSV crudo manteniendo todo como strings para analizar
df_raw = pd.read_csv(RAW, dtype=str, keep_default_na=False)

print(f"Shape del dataset: {df_raw.shape}")
print(f"\nColumnas: {list(df_raw.columns)}")
print(f"\nPrimeras filas:")
df_raw.head(10)

FileNotFoundError: [Errno 2] No such file or directory: '/home/jmsa/IESRafaelAlberti/RafaelAlberti25_26/Modulos/SBD/UD1/data/sales_raw.csv'

## 3. Análisis Exploratorio de Datos (EDA)

### 3.1 Información general del dataset

In [ ]:
# Información del dataset
print("Información del dataset:")
print(f"Número de filas: {len(df_raw)}")
print(f"Número de columnas: {len(df_raw.columns)}")
print(f"\nTipos de datos (raw - todo como string):")
df_raw.info()

### 3.2 Análisis de valores únicos por columna

In [ ]:
# Valores únicos por columna
print("Valores únicos por columna:")
for col in df_raw.columns:
    n_unique = df_raw[col].nunique()
    n_empty = (df_raw[col] == '').sum()
    print(f"{col:15s} | Únicos: {n_unique:6d} | Vacíos: {n_empty:6d}")

### 3.3 Análisis de la columna CANAL (problemas de consistencia)

In [ ]:
# Análisis del campo canal
print("Valores únicos en CANAL (sin normalizar):")
print(df_raw['canal'].value_counts())
print(f"\nTotal de valores únicos: {df_raw['canal'].nunique()}")

### 3.4 Análisis de campos numéricos (precio, unidades, importe)

In [ ]:
# Muestra de valores en precio (puede tener comas como separador decimal)
print("Muestra de valores en PRECIO:")
print(df_raw['precio'].head(20).tolist())

print("\nMuestra de valores en IMPORTE:")
print(df_raw['importe'].head(20).tolist())

print("\nMuestra de valores en UNIDADES:")
print(df_raw['unidades'].head(20).tolist())

### 3.5 Análisis de fechas

In [ ]:
# Análisis de fechas
print("Muestra de fechas:")
print(df_raw['fecha'].head(20).tolist())

print("\nValores únicos de fechas (primeros 20):")
print(sorted(df_raw['fecha'].unique())[:20])

## 4. Limpieza y normalización de datos

In [ ]:
# Crear una copia para trabajar
df = df_raw.copy()

# Limpiar espacios en blanco
for col in ["fecha", "tienda_id", "sku", "canal", "precio", "importe"]:
    df[col] = df[col].astype(str).str.strip()

print("Datos después de limpiar espacios en blanco")

In [ ]:
# Convertir unidades a numérico
df["unidades"] = pd.to_numeric(df["unidades"], errors="coerce")

print("Estadísticas de UNIDADES después de conversión:")
print(df["unidades"].describe())
print(f"\nValores nulos: {df['unidades'].isna().sum()}")
print(f"Valores negativos: {(df['unidades'] < 0).sum()}")

In [ ]:
# Normalizar decimales en precio (coma a punto) y convertir a numérico
df["precio"] = df["precio"].str.replace(",", ".", regex=False)
df["precio"] = pd.to_numeric(df["precio"], errors="coerce")

print("Estadísticas de PRECIO después de conversión:")
print(df["precio"].describe())
print(f"\nValores nulos: {df['precio'].isna().sum()}")
print(f"Valores <= 0: {(df['precio'] <= 0).sum()}")

In [ ]:
# Normalizar decimales en importe (coma a punto) y convertir a numérico
df["importe"] = df["importe"].str.replace(",", ".", regex=False)
df["importe"] = pd.to_numeric(df["importe"], errors="coerce")

print("Estadísticas de IMPORTE después de conversión:")
print(df["importe"].describe())
print(f"\nValores nulos: {df['importe'].isna().sum()}")

In [ ]:
# Convertir fechas
df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")

print("Estadísticas de FECHA después de conversión:")
print(f"Valores nulos: {df['fecha'].isna().sum()}")
print(f"Fecha mínima: {df['fecha'].min()}")
print(f"Fecha máxima: {df['fecha'].max()}")
print(f"\nFechas futuras (> hoy): {(df['fecha'] > pd.Timestamp.today()).sum()}")

In [ ]:
# Normalizar canal
canon = {
    "web": "web",
    "tienda": "tienda",
    "app": "app",
    "web ": "web",
    " wEb ": "web",
    "Web": "web",
    "WEB": "web",
    "online": "web",
    "W": "web"
}

df["canal_norm"] = df["canal"].str.lower().str.strip().map(canon).fillna(df["canal"].str.lower().str.strip())

print("Distribución de CANAL normalizado:")
print(df["canal_norm"].value_counts())
print(f"\nCanales válidos (tienda, web, app): {df['canal_norm'].isin(['tienda', 'web', 'app']).sum()}")
print(f"Canales inválidos: {(~df['canal_norm'].isin(['tienda', 'web', 'app'])).sum()}")

## 5. Cálculo de Medidas de Calidad (ANTES de limpieza profunda)

### 5.1 Completitud (Completeness)

Mide el porcentaje de valores no nulos en campos críticos.

In [ ]:
def compute_completeness(df, total):
    """Calcula métricas de completitud"""
    def ratio(x):
        return round(float(x) / total, 4) if total > 0 else 0.0
    
    metrics = {
        "precio": {"value": ratio(df["precio"].notna().sum()), "threshold": 0.99},
        "unidades": {"value": ratio(df["unidades"].notna().sum()), "threshold": 0.99},
        "importe": {"value": ratio(df["importe"].notna().sum()), "threshold": 0.99},
    }
    return metrics

total_rows = len(df)
completeness = compute_completeness(df, total_rows)

print("=" * 60)
print("COMPLETITUD (Completeness)")
print("=" * 60)
for field, data in completeness.items():
    value_pct = data["value"] * 100
    threshold_pct = data["threshold"] * 100
    status = "✓ OK" if data["value"] >= data["threshold"] else "✗ FALLA"
    print(f"{field:15s} | {value_pct:6.2f}% | Umbral: {threshold_pct:6.2f}% | {status}")
    
print(f"\nTotal de registros: {total_rows}")

### 5.2 Unicidad (Uniqueness)

Verifica si la clave primaria (fecha, tienda_id, sku) es única.

In [ ]:
def compute_uniqueness(df, total):
    """Calcula métricas de unicidad"""
    def ratio(x):
        return round(float(x) / total, 4) if total > 0 else 0.0
    
    # Crear clave compuesta
    key = df[["fecha", "tienda_id", "sku"]].astype(str).agg("|".join, axis=1)
    duplicated = key.duplicated(keep=False)
    dup_rate = duplicated.mean()
    
    metrics = {
        "fecha_tienda_sku": {
            "value": round(1.0 - float(dup_rate), 4),
            "threshold": 1.0
        }
    }
    
    return metrics, duplicated

uniqueness, duplicated_mask = compute_uniqueness(df, total_rows)

print("=" * 60)
print("UNICIDAD (Uniqueness)")
print("=" * 60)
for field, data in uniqueness.items():
    value_pct = data["value"] * 100
    threshold_pct = data["threshold"] * 100
    status = "✓ OK" if data["value"] >= data["threshold"] else "✗ FALLA"
    print(f"{field:25s} | {value_pct:6.2f}% | Umbral: {threshold_pct:6.2f}% | {status}")

print(f"\nRegistros duplicados: {duplicated_mask.sum()}")
print(f"Registros únicos: {total_rows - duplicated_mask.sum()}")

In [ ]:
# Mostrar ejemplos de duplicados
if duplicated_mask.sum() > 0:
    print("\nEjemplos de registros duplicados:")
    dup_df = df[duplicated_mask].sort_values(["fecha", "tienda_id", "sku"]).head(10)
    display(dup_df[["fecha", "tienda_id", "sku", "canal", "unidades", "precio", "importe"]])

### 5.3 Validez (Validity)

Verifica que los valores estén dentro de dominios válidos.

In [ ]:
def compute_validity(df, total):
    """Calcula métricas de validez"""
    def ratio(x):
        return round(float(x) / total, 4) if total > 0 else 0.0
    
    # Canal válido
    valid_canal = df["canal_norm"].isin(["tienda", "web", "app"])
    
    # Precio > 0
    valid_precio = (df["precio"] > 0) & df["precio"].notna()
    
    # Unidades >= 0
    valid_unidades = (df["unidades"] >= 0) & df["unidades"].notna()
    
    # Fecha razonable (no nula y <= hoy)
    valid_fecha = df["fecha"].notna() & (df["fecha"] <= pd.Timestamp.today())
    
    metrics = {
        "canal_in_set": {"value": ratio(valid_canal.sum()), "threshold": 1.0},
        "precio_gt_0": {"value": ratio(valid_precio.sum()), "threshold": 0.99},
        "unidades_ge_0": {"value": ratio(valid_unidades.sum()), "threshold": 0.99},
        "fecha_valida": {"value": ratio(valid_fecha.sum()), "threshold": 1.0},
    }
    
    return metrics, {
        "canal": valid_canal,
        "precio": valid_precio,
        "unidades": valid_unidades,
        "fecha": valid_fecha
    }

validity, validity_masks = compute_validity(df, total_rows)

print("=" * 60)
print("VALIDEZ (Validity)")
print("=" * 60)
for field, data in validity.items():
    value_pct = data["value"] * 100
    threshold_pct = data["threshold"] * 100
    status = "✓ OK" if data["value"] >= data["threshold"] else "✗ FALLA"
    print(f"{field:20s} | {value_pct:6.2f}% | Umbral: {threshold_pct:6.2f}% | {status}")

In [ ]:
# Detalles de problemas de validez
print("\nDETALLE DE PROBLEMAS DE VALIDEZ:")
print(f"Canales inválidos: {(~validity_masks['canal']).sum()}")
print(f"Precios inválidos (<=0 o nulos): {(~validity_masks['precio']).sum()}")
print(f"Unidades inválidas (<0 o nulas): {(~validity_masks['unidades']).sum()}")
print(f"Fechas inválidas (nulas o futuras): {(~validity_masks['fecha']).sum()}")

In [ ]:
# Ejemplos de valores inválidos
print("\nEjemplos de CANALES INVÁLIDOS:")
if (~validity_masks['canal']).sum() > 0:
    print(df[~validity_masks['canal']][['canal', 'canal_norm']].drop_duplicates().head(10))
else:
    print("No hay canales inválidos")

In [ ]:
print("\nEjemplos de UNIDADES NEGATIVAS:")
if ((df['unidades'] < 0) & df['unidades'].notna()).sum() > 0:
    display(df[(df['unidades'] < 0) & df['unidades'].notna()].head(10))
else:
    print("No hay unidades negativas")

In [ ]:
print("\nEjemplos de FECHAS FUTURAS:")
if (df['fecha'] > pd.Timestamp.today()).sum() > 0:
    display(df[df['fecha'] > pd.Timestamp.today()].head(10))
else:
    print("No hay fechas futuras")

### 5.4 Consistencia (Consistency)

Verifica que `importe ≈ unidades × precio` (tolerancia ±2%)

In [ ]:
def compute_consistency(df, total):
    """Calcula métricas de consistencia"""
    def ratio(x):
        return round(float(x) / total, 4) if total > 0 else 0.0
    
    # Filtrar registros con datos completos
    test = df.dropna(subset=["unidades", "precio", "importe"])
    
    if len(test) == 0:
        return {"importe_eq_unidades_x_precio_tol2pct": {"value": 0.0, "threshold": 0.98}}, None
    
    # Calcular el importe esperado
    expected_importe = test["unidades"] * test["precio"]
    actual_importe = test["importe"]
    
    # Calcular diferencia
    diff = (actual_importe - expected_importe).abs()
    tolerance = 0.02 * (expected_importe.abs() + 1e-9)
    
    # Registros consistentes
    consistent = diff <= tolerance
    
    metrics = {
        "importe_eq_unidades_x_precio_tol2pct": {
            "value": ratio(consistent.sum()),
            "threshold": 0.98
        }
    }
    
    return metrics, consistent

consistency, consistency_mask = compute_consistency(df, total_rows)

print("=" * 60)
print("CONSISTENCIA (Consistency)")
print("=" * 60)
for field, data in consistency.items():
    value_pct = data["value"] * 100
    threshold_pct = data["threshold"] * 100
    status = "✓ OK" if data["value"] >= data["threshold"] else "✗ FALLA"
    print(f"{field:40s} | {value_pct:6.2f}% | Umbral: {threshold_pct:6.2f}% | {status}")

In [ ]:
# Analizar registros inconsistentes
if consistency_mask is not None:
    test_df = df.dropna(subset=["unidades", "precio", "importe"]).copy()
    test_df["expected_importe"] = test_df["unidades"] * test_df["precio"]
    test_df["diff"] = (test_df["importe"] - test_df["expected_importe"]).abs()
    test_df["diff_pct"] = (test_df["diff"] / (test_df["expected_importe"].abs() + 1e-9)) * 100
    
    inconsistent_df = test_df[~consistency_mask]
    
    print(f"\nRegistros con datos completos: {len(test_df)}")
    print(f"Registros consistentes: {consistency_mask.sum()}")
    print(f"Registros inconsistentes: {len(inconsistent_df)}")
    
    if len(inconsistent_df) > 0:
        print("\nEjemplos de registros INCONSISTENTES:")
        cols = ["fecha", "tienda_id", "sku", "unidades", "precio", "importe", "expected_importe", "diff", "diff_pct"]
        display(inconsistent_df[cols].head(10))

### 5.5 Puntualidad (Timeliness)

Mide cuán recientes son los datos.

In [ ]:
def compute_timeliness(df):
    """Calcula métricas de puntualidad"""
    max_date = df["fecha"].max()
    
    if pd.isna(max_date):
        return {
            "max_date": None,
            "lag_days": None,
            "sla": "D+1 10:00"
        }
    
    lag_days = (pd.Timestamp.today().normalize() - max_date).days
    
    return {
        "max_date": str(max_date.date()),
        "lag_days": int(lag_days),
        "sla": "D+1 10:00"
    }

timeliness = compute_timeliness(df)

print("=" * 60)
print("PUNTUALIDAD (Timeliness)")
print("=" * 60)
print(f"Fecha máxima en datos: {timeliness['max_date']}")
print(f"Días de retraso: {timeliness['lag_days']}")
print(f"SLA definido: {timeliness['sla']}")
print(f"Fecha de hoy: {pd.Timestamp.today().date()}")

## 6. Resumen de Medidas de Calidad

In [ ]:
# Crear reporte de calidad estructurado
quality_metrics_before = OrderedDict()
quality_metrics_before["completeness"] = completeness
quality_metrics_before["uniqueness"] = uniqueness
quality_metrics_before["validity"] = validity
quality_metrics_before["consistency"] = consistency
quality_metrics_before["timeliness"] = timeliness

print("=" * 60)
print("RESUMEN DE MEDIDAS DE CALIDAD (ANTES DE LIMPIEZA)")
print("=" * 60)
print(f"\nTotal de registros analizados: {total_rows}\n")

# Calcular puntuación general
all_checks = []

for dimension, metrics in quality_metrics_before.items():
    if dimension == "timeliness":
        continue  # Puntualidad no tiene umbral numérico
    
    print(f"\n{dimension.upper()}:")
    for metric, data in metrics.items():
        if isinstance(data, dict) and "value" in data:
            value_pct = data["value"] * 100
            threshold_pct = data["threshold"] * 100
            passed = data["value"] >= data["threshold"]
            all_checks.append(passed)
            status = "✓" if passed else "✗"
            print(f"  {status} {metric:40s} | {value_pct:6.2f}% (umbral: {threshold_pct:6.2f}%)")

print(f"\n{'-'*60}")
checks_passed = sum(all_checks)
total_checks = len(all_checks)
quality_score = (checks_passed / total_checks * 100) if total_checks > 0 else 0
print(f"PUNTUACIÓN DE CALIDAD: {quality_score:.1f}% ({checks_passed}/{total_checks} checks pasados)")
print("=" * 60)

## 7. Tabla resumen de problemas detectados

In [ ]:
# Crear tabla resumen de problemas
problemas = []

# Completitud
for field, data in completeness.items():
    missing = int(total_rows * (1 - data["value"]))
    if missing > 0:
        problemas.append({
            "Dimensión": "Completitud",
            "Problema": f"{field} con valores nulos",
            "Registros afectados": missing,
            "Porcentaje": f"{(1-data['value'])*100:.2f}%"
        })

# Unicidad
dup_count = duplicated_mask.sum()
if dup_count > 0:
    problemas.append({
        "Dimensión": "Unicidad",
        "Problema": "Registros duplicados (fecha, tienda, sku)",
        "Registros afectados": int(dup_count),
        "Porcentaje": f"{(dup_count/total_rows)*100:.2f}%"
    })

# Validez
for field, mask in validity_masks.items():
    invalid_count = (~mask).sum()
    if invalid_count > 0:
        problemas.append({
            "Dimensión": "Validez",
            "Problema": f"{field} con valores inválidos",
            "Registros afectados": int(invalid_count),
            "Porcentaje": f"{(invalid_count/total_rows)*100:.2f}%"
        })

# Consistencia
if consistency_mask is not None:
    inconsistent_count = (~consistency_mask).sum()
    if inconsistent_count > 0:
        problemas.append({
            "Dimensión": "Consistencia",
            "Problema": "importe ≠ unidades × precio (±2%)",
            "Registros afectados": int(inconsistent_count),
            "Porcentaje": f"{(inconsistent_count/total_rows)*100:.2f}%"
        })

df_problemas = pd.DataFrame(problemas)

print("\n" + "=" * 80)
print("TABLA RESUMEN DE PROBLEMAS DE CALIDAD DETECTADOS")
print("=" * 80)
if len(df_problemas) > 0:
    display(df_problemas)
    print(f"\nTotal de tipos de problemas detectados: {len(df_problemas)}")
else:
    print("No se detectaron problemas de calidad.")

## 8. Exportar métricas de calidad

In [ ]:
# Crear estructura de reporte
quality_report = {
    "dataset": "sales_raw",
    "version": datetime.now().strftime("%Y_%m_%d"),
    "generated_at": datetime.now().isoformat(timespec="seconds"),
    "summary": {
        "total_rows": int(total_rows),
        "total_columns": int(len(df.columns)),
        "quality_score": round(quality_score, 2),
        "checks_passed": int(checks_passed),
        "total_checks": int(total_checks)
    },
    "metrics": {
        "before": quality_metrics_before
    },
    "problems_detected": problemas,
    "notes": "Este reporte contiene las métricas de calidad ANTES de realizar limpieza profunda."
}

# Guardar en JSON
output_file = BASE / "quality_metrics_before.json"
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(quality_report, f, indent=2, ensure_ascii=False)

print(f"✓ Métricas de calidad exportadas a: {output_file}")

## 9. Exportar tabla de problemas a CSV

In [ ]:
# Guardar tabla de problemas en CSV
if len(df_problemas) > 0:
    csv_file = BASE / "quality_problems_summary.csv"
    df_problemas.to_csv(csv_file, index=False, encoding="utf-8")
    print(f"✓ Tabla de problemas exportada a: {csv_file}")
else:
    print("No hay problemas para exportar.")

## 10. Conclusiones

Este notebook ha realizado un análisis completo de calidad de datos sobre el dataset `sales_raw.csv`, calculando las siguientes dimensiones:

1. **Completitud**: Porcentaje de valores no nulos en campos críticos
2. **Unicidad**: Verificación de claves primarias únicas
3. **Validez**: Valores dentro de dominios válidos
4. **Consistencia**: Coherencia entre campos relacionados
5. **Puntualidad**: Recencia de los datos

Los resultados han sido exportados a:
- `quality_metrics_before.json`: Reporte completo de métricas
- `quality_problems_summary.csv`: Tabla resumen de problemas detectados

### Próximos pasos:

1. **Limpieza de datos**: Aplicar transformaciones para corregir problemas detectados
2. **Publicación en Parquet**: Exportar datos limpios en formato Parquet particionado
3. **Verificación con DuckDB**: Ejecutar checks SQL sobre los datos limpios
4. **Informe de calidad**: Documentar decisiones y resultados